In [1]:
pip install ydata-profiling[pyspark]

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pyspark.sql import SparkSession
from ydata_profiling import ProfileReport

In [2]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Python Spark profiling example") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

In [4]:
# Copy File to bronze layer
from os import PathLike
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

from_path = "./the_oscar_award.csv"
to_path = "/demo/bronze/the_oscar_award.csv"

client.delete(to_path)
client.upload(to_path, from_path)

'/demo/bronze/the_oscar_award.csv'

In [5]:
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/the_oscar_award.csv"

In [6]:
# Create a DataFrame from JSON data (automatically infer schema and data types)
# There are other file formats you can read from (e.g., csv, orc, parquet)
# https://spark.apache.org/docs/2.2.0/sql-programming-guide.html#data-sources

# Read Oscar data to a dataframe
the_oscar_award_df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

In [7]:
# Note that some profiling operations can resulte in errors due to bad loading options. 
# It is a good praticce start by inspect the schema and a data sample. 
the_oscar_award_df.printSchema()
the_oscar_award_df.show()

root
 |-- year_film: integer (nullable = true)
 |-- year_ceremony: integer (nullable = true)
 |-- ceremony: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- canon_category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- film: string (nullable = true)
 |-- winner: boolean (nullable = true)

+---------+-------------+--------+--------------------+--------------------+--------------------+--------------------+------+
|year_film|year_ceremony|ceremony|            category|      canon_category|                name|                film|winner|
+---------+-------------+--------+--------------------+--------------------+--------------------+--------------------+------+
|     1927|         1928|       1|               ACTOR|ACTOR IN A LEADIN...| Richard Barthelmess|           The Noose| false|
|     1927|         1928|       1|               ACTOR|ACTOR IN A LEADIN...| Richard Barthelmess|The Patent Leathe...| false|
|     1927|         1928|       1|    

In [20]:
the_oscar_award_df.select("film").distinct().count()

11110

In [22]:
from pyspark.sql.functions import col, when, count

missing_film = the_oscar_award_df.select(
    count(when(col("film").isNull() | (col("film") == ""), "film")).alias("missing_film")
).collect()[0]["missing_film"]

print(missing_film)

359


In [10]:
# In case of error select a subset of columns until you find the column that causes that.
#For start we can use describe as starting point for data profiling
#For this example the column summary was removed due to a conflit with the first describe column "summary"
the_oscar_award_df.describe(['year_film','year_ceremony','ceremony','category','canon_category','name','film','winner']).toPandas()

,summary,year_film,year_ceremony,ceremony,category,canon_category,name,film
0,count,11110,11110,11110,11110,11110,11103,10751
1,mean,1977.1837083708372,1978.1837083708372,50.20927092709271,None,None,None,1612.8333333333333
2,stddev,27.679615879290296,27.679615879290296,27.636153746267716,None,None,None,734.0176912135651
3,min,1927,1928,1,ACTOR,ACTOR IN A LEADING ROLE,"""Ahmir """"Questlove"""" Thompson/Joseph Patel/Rob...","""Maggie Simpson in """"The Longest Daycare"""""""
4,max,2024,2025,97,WRITING (Title Writing),WRITING (Title Writing),the Technicolor Company,Éramos Pocos (One Too Many)


In [11]:
#Select the columns to profile. 
df_to_profile = the_oscar_award_df.select("year_film","year_ceremony","ceremony","category","canon_category","name","film","winner")

In [12]:
#create the Profile report using the ydata profiling tool. More info at https://docs.profiling.ydata.ai/
report = ProfileReport(df_to_profile)

In [13]:
#save profiling report in a file
report.to_file('oscar_awards.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
#close spark session
spark.stop()